# 07 - Class Imbalance: When 99% Accuracy Means Nothing

**Difficulty:** ⭐⭐⭐⭐ (Advanced)
**Estimated Time:** 3 hours
**Domain:** Credit Card Fraud Detection

---

## Why Does This Matter for ML?

Imagine you build a fraud detection model. You train it, evaluate it, and it achieves **99.2% accuracy**. You celebrate. You deploy it. Fraudsters steal millions.

What went wrong? The model learned to say "not fraud" for every single transaction — because 99.2% of transactions ARE legitimate. It never detected a single fraud case.

This is the **class imbalance problem**, and it affects almost every real-world classification task:
- Fraud detection: 0.1–2% fraud
- Medical diagnosis: 1–5% rare disease
- Equipment failure: <1% defective parts
- Spam detection: 1–5% spam

**In this notebook you will learn:**
- Why accuracy is a broken metric for imbalanced data
- How to properly evaluate imbalanced classifiers (precision, recall, F1, AUC-ROC)
- Four strategies to fix imbalance: undersampling, random oversampling, SMOTE, class weights
- The critical mistake of applying SMOTE before splitting data
- How to compare strategies fairly

---

## The Real-World Analogy: The Airport Metal Detector

Picture airport security in a small regional airport. Suppose 1 in 10,000 passengers carries prohibited metal. You design a metal detector system.

**Your naive system:** Always says "no metal detected."  
**Your accuracy:** 9,999 / 10,000 = **99.99%**  
**Actual usefulness:** Zero. You catch zero threats.

A real metal detector optimizes for **recall** (catching all metal) even if it occasionally produces false alarms (false positives). It accepts lower "accuracy" to be actually useful.

The same principle applies to fraud detection: **we'd rather flag a few legitimate transactions for review than miss any fraud.**

## Setup: Import Libraries

We need:
- **numpy / pandas**: data manipulation
- **matplotlib / seaborn**: visualizations
- **sklearn**: machine learning models and metrics
- **imblearn**: imbalanced-learn library with SMOTE and resampling tools

Install if needed: `pip install imbalanced-learn`

In [ ]:
# Core data science libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')  # Suppress convergence warnings for cleaner output

# Sklearn: models, metrics, preprocessing
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)
from sklearn.decomposition import PCA  # For 2D visualization of SMOTE effect
from sklearn.datasets import make_classification

# Imbalanced-learn: resampling strategies
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline  # Handles resampling inside CV

# Set consistent random seed so results are reproducible
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('All libraries loaded successfully!')
print(f'NumPy: {np.__version__}, Pandas: {pd.__version__}')

## Step 1: Create Synthetic Credit Card Fraud Dataset

We'll build a realistic dataset with:
- **10,000 transactions** total
- **2% fraud rate** (200 fraud cases, 9,800 legitimate)
- Features inspired by real credit card data: amount, time, merchant category scores, velocity features

The **2% fraud rate** is actually generous — real datasets often have <0.5% fraud.

In [ ]:
def create_fraud_dataset(n_legitimate=9800, n_fraud=200, random_state=42):
    """
    Creates a synthetic credit card transaction dataset.
    Fraud cases have distinctive statistical patterns vs legitimate ones.
    """
    rng = np.random.RandomState(random_state)
    
    # --- LEGITIMATE TRANSACTIONS ---
    # Transaction amounts: log-normal distribution (most transactions are small)
    legit_amount = rng.lognormal(mean=3.5, sigma=1.2, size=n_legitimate)  # Mean ~$33
    # Time feature: hour of day (0-23), legitimate transactions peak during business hours
    legit_hour = rng.normal(loc=14, scale=4, size=n_legitimate).clip(0, 23)
    # Merchant category risk score (0-1): most legit transactions at low-risk merchants
    legit_merchant_risk = rng.beta(a=2, b=8, size=n_legitimate)  # Skewed toward 0
    # Number of transactions in last 24h: moderate for legit users
    legit_velocity = rng.poisson(lam=3, size=n_legitimate).clip(0, 20)
    # Distance from home location (km): usually close for legit transactions
    legit_distance = rng.exponential(scale=20, size=n_legitimate).clip(0, 500)
    # Card-not-present flag: 30% of legit are online (card not physically present)
    legit_cnp = rng.binomial(n=1, p=0.3, size=n_legitimate)
    # PCA-derived features (representing compressed transaction patterns)
    legit_v1 = rng.normal(0.2, 1.0, n_legitimate)
    legit_v2 = rng.normal(-0.1, 0.9, n_legitimate)
    legit_v3 = rng.normal(0.0, 1.1, n_legitimate)

    # --- FRAUDULENT TRANSACTIONS ---
    # Fraud amounts: bimodal — many small "test" transactions AND some large ones
    fraud_amount_small = rng.uniform(1, 10, size=n_fraud // 2)    # Small test charges
    fraud_amount_large = rng.uniform(200, 2000, size=n_fraud // 2)  # Large purchases
    fraud_amount = np.concatenate([fraud_amount_small, fraud_amount_large])
    # Fraud happens more at odd hours (night/early morning)
    fraud_hour = rng.normal(loc=3, scale=3, size=n_fraud).clip(0, 23)
    # Fraudsters often use high-risk merchant categories
    fraud_merchant_risk = rng.beta(a=6, b=3, size=n_fraud)  # Skewed toward 1
    # High velocity: fraudsters make many transactions quickly
    fraud_velocity = rng.poisson(lam=8, size=n_fraud).clip(0, 30)
    # Large distance: card used far from home
    fraud_distance = rng.exponential(scale=150, size=n_fraud).clip(0, 2000)
    # Fraud is almost always card-not-present (online)
    fraud_cnp = rng.binomial(n=1, p=0.85, size=n_fraud)
    # PCA features shifted for fraud class
    fraud_v1 = rng.normal(-2.5, 1.2, n_fraud)
    fraud_v2 = rng.normal(1.8, 1.0, n_fraud)
    fraud_v3 = rng.normal(-1.5, 0.8, n_fraud)

    # Combine legitimate and fraudulent transactions into one DataFrame
    df = pd.DataFrame({
        'amount': np.concatenate([legit_amount, fraud_amount]),
        'hour': np.concatenate([legit_hour, fraud_hour]),
        'merchant_risk': np.concatenate([legit_merchant_risk, fraud_merchant_risk]),
        'velocity_24h': np.concatenate([legit_velocity, fraud_velocity]),
        'distance_km': np.concatenate([legit_distance, fraud_distance]),
        'card_not_present': np.concatenate([legit_cnp, fraud_cnp]),
        'v1': np.concatenate([legit_v1, fraud_v1]),
        'v2': np.concatenate([legit_v2, fraud_v2]),
        'v3': np.concatenate([legit_v3, fraud_v3]),
        'is_fraud': np.concatenate([np.zeros(n_legitimate), np.ones(n_fraud)])
    })

    # Shuffle rows so fraud cases aren't all at the end
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    return df

# Create the dataset
df = create_fraud_dataset(n_legitimate=9800, n_fraud=200)

print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['is_fraud'].value_counts())
print(f'\nFraud rate: {df["is_fraud"].mean()*100:.2f}%')
print(f'\nFeature summary:')
df.describe().round(2)

## Step 2: Visualize the Imbalance

Before building any model, always visualize your class distribution. The imbalance is immediately apparent.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# --- Plot 1: Class Distribution Bar Chart ---
class_counts = df['is_fraud'].value_counts()
colors = ['#2ecc71', '#e74c3c']  # Green = legit, Red = fraud
bars = axes[0].bar(['Legitimate\n(0)', 'Fraud\n(1)'],
                    [class_counts[0], class_counts[1]],
                    color=colors, edgecolor='black', linewidth=0.8)

# Add count labels on top of each bar
for bar, count in zip(bars, [class_counts[0], class_counts[1]]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'{count:,}\n({count/len(df)*100:.1f}%)',
                 ha='center', va='bottom', fontweight='bold', fontsize=11)

axes[0].set_title('Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Transactions')
axes[0].set_ylim(0, 11000)

# --- Plot 2: Transaction Amount by Class ---
# Use log scale for amount because of extreme right skew
legit_amounts = df[df['is_fraud'] == 0]['amount']
fraud_amounts = df[df['is_fraud'] == 1]['amount']

axes[1].hist(np.log1p(legit_amounts), bins=40, alpha=0.7, color='#2ecc71',
             label=f'Legitimate (n={len(legit_amounts):,})', density=True)
axes[1].hist(np.log1p(fraud_amounts), bins=20, alpha=0.7, color='#e74c3c',
             label=f'Fraud (n={len(fraud_amounts):,})', density=True)

axes[1].set_title('Transaction Amount Distribution\n(log scale)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(1 + Amount)')
axes[1].set_ylabel('Density')
axes[1].legend()

# --- Plot 3: Transaction Hour by Class ---
axes[2].hist(df[df['is_fraud'] == 0]['hour'], bins=24, alpha=0.7,
             color='#2ecc71', label='Legitimate', density=True)
axes[2].hist(df[df['is_fraud'] == 1]['hour'], bins=24, alpha=0.7,
             color='#e74c3c', label='Fraud', density=True)
axes[2].set_title('Transaction Hour Distribution', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Hour of Day (0 = midnight)')
axes[2].set_ylabel('Density')
axes[2].legend()

plt.suptitle('Credit Card Fraud Dataset — Exploratory Analysis', fontsize=15,
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('\nKey observations:')
print(f'  - Class ratio: {class_counts[0]}:{class_counts[1]} (legitimate:fraud)')
print(f'  - Fraud transactions tend to occur at night (mean hour: {df[df["is_fraud"]==1]["hour"].mean():.1f})')
print(f'  - Legit transactions are mostly during the day (mean hour: {df[df["is_fraud"]==0]["hour"].mean():.1f})')

## Step 3: The Accuracy Paradox — Why Accuracy Fails

### Plain English First

"Accuracy" counts how often you're right. But if your class distribution is 98:2, then a model that **always says 'not fraud'** gets 98% accuracy without learning anything.

### The Better Metrics

| Metric | Formula | What it measures | For fraud detection |
|--------|---------|-----------------|--------------------|
| **Precision** | TP / (TP + FP) | Of flagged frauds, how many are real? | Avoid harassing good customers |
| **Recall** | TP / (TP + FN) | Of all real frauds, how many did we catch? | The critical metric — missing fraud is costly |
| **F1** | 2 × (P×R)/(P+R) | Balance between precision and recall | Overall minority class performance |
| **AUC-ROC** | Area under ROC curve | Model's ability to rank fraud above legit | Threshold-independent summary |

**Recall** is paramount in fraud detection: a missed fraud (FN) costs the bank real money. A false alarm (FP) just means a quick call to the customer.

In [ ]:
# Prepare features and target
FEATURE_COLS = ['amount', 'hour', 'merchant_risk', 'velocity_24h',
                'distance_km', 'card_not_present', 'v1', 'v2', 'v3']
TARGET_COL = 'is_fraud'

X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

# Stratified split: ensures BOTH splits have same class ratio
# Without stratify=y, you might get 0 fraud cases in test set by bad luck!
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f'Training set: {X_train.shape[0]} samples')
print(f'  - Legitimate: {(y_train == 0).sum()} ({(y_train == 0).mean()*100:.1f}%)')
print(f'  - Fraud:      {(y_train == 1).sum()} ({(y_train == 1).mean()*100:.1f}%)')
print(f'\nTest set: {X_test.shape[0]} samples')
print(f'  - Legitimate: {(y_test == 0).sum()} ({(y_test == 0).mean()*100:.1f}%)')
print(f'  - Fraud:      {(y_test == 1).sum()} ({(y_test == 1).mean()*100:.1f}%)')

# Scale features: logistic regression is sensitive to feature scale
# StandardScaler transforms each feature to mean=0, std=1
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # Fit on training data only
X_test_scaled = scaler.transform(X_test)         # Apply same transformation to test

# Helper function to compute and display all relevant metrics
def evaluate_model(y_true, y_pred, y_prob=None, model_name='Model'):
    """Returns a dict of metrics for the FRAUD class (class 1)."""
    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall':    recall_score(y_true, y_pred, zero_division=0),
        'f1':        f1_score(y_true, y_pred, zero_division=0),
    }
    # AUC-ROC requires probability scores, not just binary predictions
    if y_prob is not None:
        metrics['auc_roc'] = roc_auc_score(y_true, y_prob)
    else:
        metrics['auc_roc'] = None
    return metrics

print('\nData prepared and scaler fitted.')

In [ ]:
# ============================================================
# BASELINE: Naive Logistic Regression — No Imbalance Handling
# ============================================================

# Train a standard logistic regression — no special handling
baseline_model = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
baseline_model.fit(X_train_scaled, y_train)

# Predict classes and probability scores
y_pred_baseline = baseline_model.predict(X_test_scaled)
# predict_proba returns [P(class=0), P(class=1)] — we want P(fraud) = column index 1
y_prob_baseline = baseline_model.predict_proba(X_test_scaled)[:, 1]

baseline_metrics = evaluate_model(y_test, y_pred_baseline, y_prob_baseline, 'Baseline')

print('=' * 55)
print('BASELINE: Logistic Regression (No Imbalance Handling)')
print('=' * 55)
print(f"Accuracy:  {baseline_metrics['accuracy']:.4f}  <-- looks great, right?")
print(f"Precision: {baseline_metrics['precision']:.4f}  (for fraud class)")
print(f"Recall:    {baseline_metrics['recall']:.4f}  <-- catches almost NO fraud!")
print(f"F1-Score:  {baseline_metrics['f1']:.4f}")
print(f"AUC-ROC:   {baseline_metrics['auc_roc']:.4f}")
print()
print('Full Classification Report:')
print(classification_report(y_test, y_pred_baseline,
                            target_names=['Legitimate', 'Fraud'],
                            zero_division=0))
print()
print('>>> The model predicts almost every transaction as legitimate!')
print(f'>>> Total fraud in test set: {y_test.sum()}')
print(f'>>> Fraud correctly detected: {((y_pred_baseline == 1) & (y_test == 1)).sum()}')

# Count how many predictions were "fraud" vs "not fraud"
unique, counts = np.unique(y_pred_baseline, return_counts=True)
print(f'>>> Prediction breakdown: {dict(zip(["Legit", "Fraud"], counts))}')

## Step 4: Strategy 1 — Random Undersampling

### Plain English
We have 9,800 legitimate and 200 fraud transactions. Undersampling **randomly removes** legitimate transactions until the classes are closer in size.

**Pro:** Fast, simple, reduces training time  
**Con:** We throw away real data — potentially useful patterns in the discarded transactions are lost

**`sampling_strategy=0.5`** means: after resampling, the minority class will be 50% the size of the majority class. So if we have 200 fraud, we keep 400 legitimate → 600 total samples.

Think of it like: you have 9,800 customer receipts filed, but only 200 fraud reports to compare. Instead of finding patterns across all receipts, you randomly pick just 400 receipts to work with.

In [ ]:
# ============================================================
# STRATEGY 1: Random Undersampling
# ============================================================

# sampling_strategy=0.5 means: minority will be 50% the size of majority after resampling
# So: 200 fraud / X legit = 0.5 → X = 400 legitimate samples kept
undersampler = RandomUnderSampler(sampling_strategy=0.5, random_state=RANDOM_STATE)

# CRITICAL: Apply ONLY to training data — never touch test data!
X_train_under, y_train_under = undersampler.fit_resample(X_train_scaled, y_train)

print('UNDERSAMPLING RESULTS:')
print(f'  Before: {(y_train==0).sum()} legitimate, {(y_train==1).sum()} fraud')
print(f'  After:  {(y_train_under==0).sum()} legitimate, {(y_train_under==1).sum()} fraud')
print(f'  Samples removed: {len(y_train) - len(y_train_under)}')
print(f'  New fraud %: {y_train_under.mean()*100:.1f}%')
print()

# Train logistic regression on the undersampled data
model_under = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_under.fit(X_train_under, y_train_under)

# Evaluate on the ORIGINAL test set (not resampled — test set always stays natural)
y_pred_under = model_under.predict(X_test_scaled)
y_prob_under = model_under.predict_proba(X_test_scaled)[:, 1]

metrics_under = evaluate_model(y_test, y_pred_under, y_prob_under, 'Undersampling')

print('PERFORMANCE AFTER UNDERSAMPLING:')
print(f"Accuracy:  {metrics_under['accuracy']:.4f}")
print(f"Precision: {metrics_under['precision']:.4f}")
print(f"Recall:    {metrics_under['recall']:.4f}  <-- significantly improved!")
print(f"F1-Score:  {metrics_under['f1']:.4f}")
print(f"AUC-ROC:   {metrics_under['auc_roc']:.4f}")
print()
print(classification_report(y_test, y_pred_under,
                             target_names=['Legitimate', 'Fraud'], zero_division=0))

## Step 5: Strategy 2 — Random Oversampling

### Plain English
Instead of removing majority class samples, we **duplicate** minority class (fraud) samples until the classes are balanced.

**Pro:** No information lost from majority class  
**Con:** The model sees exact duplicate fraud cases → it may memorize specific fraud cases rather than learning general patterns (overfitting)

Think of it like photocopying the 200 fraud reports 49 times each to have 9,800 fraud copies — you now have equal numbers, but every copy is identical to an original.

In [ ]:
# ============================================================
# STRATEGY 2: Random Oversampling
# ============================================================

# Default sampling_strategy='auto' makes minority class equal to majority class
# So fraud samples are duplicated from 160 to 7840 (to match legit training count)
oversampler = RandomOverSampler(sampling_strategy='auto', random_state=RANDOM_STATE)

# Apply ONLY to training data
X_train_over, y_train_over = oversampler.fit_resample(X_train_scaled, y_train)

print('RANDOM OVERSAMPLING RESULTS:')
print(f'  Before: {(y_train==0).sum()} legitimate, {(y_train==1).sum()} fraud')
print(f'  After:  {(y_train_over==0).sum()} legitimate, {(y_train_over==1).sum()} fraud')

# How many unique fraud samples do we have? Should still be 160 (just duplicated)
# Find the fraud rows in the oversampled training set
fraud_indices = np.where(y_train_over == 1)[0]
fraud_features_over = X_train_over[fraud_indices]

# unique() on a 2D array: count unique rows
unique_fraud_rows = np.unique(fraud_features_over, axis=0)
print(f'  Total fraud samples after oversampling: {len(fraud_indices)}')
print(f'  Unique fraud samples (not duplicates): {len(unique_fraud_rows)}')
print(f'  Average duplicates per original: {len(fraud_indices)/len(unique_fraud_rows):.1f}x')
print()

# Train logistic regression on oversampled data
model_over = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_over.fit(X_train_over, y_train_over)

# Evaluate on original test set
y_pred_over = model_over.predict(X_test_scaled)
y_prob_over = model_over.predict_proba(X_test_scaled)[:, 1]

metrics_over = evaluate_model(y_test, y_pred_over, y_prob_over, 'Oversampling')

print('PERFORMANCE AFTER RANDOM OVERSAMPLING:')
print(f"Accuracy:  {metrics_over['accuracy']:.4f}")
print(f"Precision: {metrics_over['precision']:.4f}")
print(f"Recall:    {metrics_over['recall']:.4f}")
print(f"F1-Score:  {metrics_over['f1']:.4f}")
print(f"AUC-ROC:   {metrics_over['auc_roc']:.4f}")
print()
print(classification_report(y_test, y_pred_over,
                             target_names=['Legitimate', 'Fraud'], zero_division=0))

## Step 6: Strategy 3 — SMOTE (Synthetic Minority Oversampling Technique)

### Plain English

Instead of photocopying fraud cases (random oversampling), SMOTE **creates brand-new, plausible fraud cases** by interpolating between existing ones.

### How SMOTE Works (Step by Step)

1. Pick a minority sample (e.g., one fraud transaction)
2. Find its **k nearest minority neighbors** (other fraud transactions that look similar)
3. Pick one of those neighbors randomly
4. Create a new sample **somewhere along the line between** the original and the neighbor
5. The new sample has realistic values — not just a copy

**Analogy:** Instead of photocopying 200 fraud reports 49 times, you study the 200 reports and **write 9,600 new, plausible fraud scenarios** based on patterns in the originals. The new scenarios are realistic variations, not copies.

### Critical Rule: SMOTE Only on Training Data

If you apply SMOTE before splitting, synthetic samples from test fraud cases end up in training data. Your model gets "hints" about the test set → evaluation is dishonestly optimistic.

In [ ]:
# ============================================================
# STRATEGY 3: SMOTE
# ============================================================

# k_neighbors=5: for each minority sample, consider 5 nearest minority neighbors
# A larger k creates smoother, more varied synthetic samples
smote = SMOTE(k_neighbors=5, random_state=RANDOM_STATE, sampling_strategy='auto')

# Apply ONLY to training data
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

print('SMOTE RESULTS:')
print(f'  Before: {(y_train==0).sum()} legitimate, {(y_train==1).sum()} fraud')
print(f'  After:  {(y_train_smote==0).sum()} legitimate, {(y_train_smote==1).sum()} fraud')

# Check uniqueness: SMOTE creates genuinely new (synthetic) samples
fraud_idx_smote = np.where(y_train_smote == 1)[0]
fraud_features_smote = X_train_smote[fraud_idx_smote]
unique_smote = np.unique(fraud_features_smote, axis=0)
print(f'  Total fraud samples: {len(fraud_idx_smote)}')
print(f'  Unique fraud samples: {len(unique_smote)} (all are different!)')
print()

# Train logistic regression on SMOTE data
model_smote = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_smote.fit(X_train_smote, y_train_smote)

# Evaluate on original test set
y_pred_smote = model_smote.predict(X_test_scaled)
y_prob_smote = model_smote.predict_proba(X_test_scaled)[:, 1]

metrics_smote = evaluate_model(y_test, y_pred_smote, y_prob_smote, 'SMOTE')

print('PERFORMANCE AFTER SMOTE:')
print(f"Accuracy:  {metrics_smote['accuracy']:.4f}")
print(f"Precision: {metrics_smote['precision']:.4f}")
print(f"Recall:    {metrics_smote['recall']:.4f}")
print(f"F1-Score:  {metrics_smote['f1']:.4f}")
print(f"AUC-ROC:   {metrics_smote['auc_roc']:.4f}")
print()
print(classification_report(y_test, y_pred_smote,
                             target_names=['Legitimate', 'Fraud'], zero_division=0))

## Step 7: Visualize SMOTE — Before and After in 2D

Our data has 9 features. We use **PCA** (Principal Component Analysis) to compress them to 2 dimensions, so we can see what SMOTE actually does to the minority class distribution.

In [ ]:
# Reduce training data to 2D using PCA for visualization
# PCA finds the 2 directions that explain the most variance
pca = PCA(n_components=2, random_state=RANDOM_STATE)

# Fit PCA on the original (pre-SMOTE) training data, then transform both
X_train_2d = pca.fit_transform(X_train_scaled)
# Apply same PCA projection to SMOTE data (not fit again)
X_train_smote_2d = pca.transform(X_train_smote)

# Identify original vs synthetic minority samples in SMOTE output
# The first len(X_train_scaled) rows of X_train_smote are the original data
# Rows after that are SMOTE-generated synthetic samples
n_original = len(X_train_scaled)
n_synthetic = len(X_train_smote) - n_original

# In the original training data, which rows are fraud?
orig_fraud_mask = y_train == 1
orig_legit_mask = y_train == 0

# In SMOTE output, synthetic fraud is the extra samples beyond the original size
# The SMOTE function appends synthetic samples at the END of the arrays
synthetic_X_2d = X_train_smote_2d[n_original:, :]  # New rows added by SMOTE
synthetic_y = y_train_smote[n_original:]            # Their labels (all = 1 = fraud)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- LEFT PLOT: Before SMOTE ---
axes[0].scatter(X_train_2d[orig_legit_mask, 0], X_train_2d[orig_legit_mask, 1],
               alpha=0.15, c='#2ecc71', s=15, label=f'Legitimate ({orig_legit_mask.sum():,})')
axes[0].scatter(X_train_2d[orig_fraud_mask, 0], X_train_2d[orig_fraud_mask, 1],
               alpha=0.8, c='#e74c3c', s=50, marker='x',
               label=f'Fraud ({orig_fraud_mask.sum()})', linewidths=1.5)
axes[0].set_title('BEFORE SMOTE\nOriginal Training Data (2D PCA)',
                  fontsize=12, fontweight='bold')
axes[0].set_xlabel('Principal Component 1')
axes[0].set_ylabel('Principal Component 2')
axes[0].legend(markerscale=1.5)

# --- RIGHT PLOT: After SMOTE ---
axes[1].scatter(X_train_2d[orig_legit_mask, 0], X_train_2d[orig_legit_mask, 1],
               alpha=0.15, c='#2ecc71', s=15, label=f'Legitimate (original {orig_legit_mask.sum():,})')
axes[1].scatter(X_train_2d[orig_fraud_mask, 0], X_train_2d[orig_fraud_mask, 1],
               alpha=0.9, c='#e74c3c', s=60, marker='x',
               label=f'Fraud (original {orig_fraud_mask.sum()})', linewidths=2)

# Synthetic samples are in the region between original fraud points
synthetic_fraud_2d = synthetic_X_2d[synthetic_y == 1]
axes[1].scatter(synthetic_fraud_2d[:, 0], synthetic_fraud_2d[:, 1],
               alpha=0.4, c='#ff8c00', s=20, marker='o',
               label=f'Synthetic Fraud (SMOTE: {len(synthetic_fraud_2d):,})')

axes[1].set_title('AFTER SMOTE\nSynthetic Fraud Samples Added (2D PCA)',
                  fontsize=12, fontweight='bold')
axes[1].set_xlabel('Principal Component 1')
axes[1].set_ylabel('Principal Component 2')
axes[1].legend(markerscale=1.5)

plt.suptitle('SMOTE: Synthetic Minority Oversampling Visualization',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'SMOTE filled the minority region with {len(synthetic_fraud_2d):,} synthetic samples.')
print('Orange dots are NOT copies — each is a new interpolated point between real fraud cases.')

## Step 8: Strategy 4 — Class Weights

### Plain English

Instead of resampling data, we tell the model: **"Getting a fraud case wrong costs you 49x more than getting a legitimate case wrong."**

The model adjusts its decision boundary to avoid costly mistakes on the minority class.

**This is the simplest strategy:** no extra data, no resampling, just one parameter change.

**`class_weight='balanced'`** automatically calculates weights as:
- Weight for class 0 (legit) = total_samples / (2 × n_legit)
- Weight for class 1 (fraud) = total_samples / (2 × n_fraud)

With 7840 legit and 160 fraud in training: fraud gets ~49x more weight.

In [ ]:
# ============================================================
# STRATEGY 4: Class Weights
# ============================================================

# class_weight='balanced': automatically compute weights inversely proportional to class frequencies
# This effectively tells the model: pay more attention to the rare class
model_weighted = LogisticRegression(
    class_weight='balanced',  # The key parameter
    max_iter=1000,
    random_state=RANDOM_STATE
)

# Train on the ORIGINAL (unmodified) training data — no resampling needed!
model_weighted.fit(X_train_scaled, y_train)

# Show the computed weights
n_train_total = len(y_train)
n_legit_train = (y_train == 0).sum()
n_fraud_train = (y_train == 1).sum()

# Sklearn's 'balanced' formula: n_samples / (n_classes * n_class_samples)
w_legit = n_train_total / (2 * n_legit_train)
w_fraud = n_train_total / (2 * n_fraud_train)

print('Computed class weights (balanced):')
print(f'  Legitimate (class 0): {w_legit:.4f}')
print(f'  Fraud      (class 1): {w_fraud:.4f}')
print(f'  Ratio: fraud is weighted {w_fraud/w_legit:.1f}x more than legitimate')
print()

# Evaluate on test set
y_pred_weighted = model_weighted.predict(X_test_scaled)
y_prob_weighted = model_weighted.predict_proba(X_test_scaled)[:, 1]

metrics_weighted = evaluate_model(y_test, y_pred_weighted, y_prob_weighted, 'Class Weights')

print('PERFORMANCE WITH CLASS WEIGHTS:')
print(f"Accuracy:  {metrics_weighted['accuracy']:.4f}")
print(f"Precision: {metrics_weighted['precision']:.4f}")
print(f"Recall:    {metrics_weighted['recall']:.4f}")
print(f"F1-Score:  {metrics_weighted['f1']:.4f}")
print(f"AUC-ROC:   {metrics_weighted['auc_roc']:.4f}")
print()
print(classification_report(y_test, y_pred_weighted,
                             target_names=['Legitimate', 'Fraud'], zero_division=0))

## Step 9: Critical Lesson — The SMOTE Data Leakage Trap

### The Wrong Way (Data Leakage)

Many beginners make this mistake: they apply SMOTE to the **entire dataset** before splitting into train/test. This causes **data leakage**:

1. SMOTE sees test fraud cases
2. Creates synthetic samples based on test fraud cases  
3. Those synthetic samples end up in training data
4. Model gets indirect information about test set → artificially inflated metrics

### The Right Way

SMOTE goes **inside** the training pipeline, **after** the train/test split.

In [ ]:
# ============================================================
# DATA LEAKAGE DEMO: Wrong Way vs. Right Way
# ============================================================

print('=' * 60)
print('WRONG WAY: SMOTE applied before train/test split (DATA LEAKAGE)')
print('=' * 60)

# WRONG: Apply SMOTE to entire dataset BEFORE splitting
X_all = df[FEATURE_COLS].values
y_all = df[TARGET_COL].values

# Scale all data first (also wrong — we're using test data to fit scaler)
scaler_wrong = StandardScaler()
X_all_scaled = scaler_wrong.fit_transform(X_all)  # Wrong: test data influences scaler

# SMOTE on entire dataset — synthetic samples are created using ALL fraud cases
# including what will become test fraud cases
smote_wrong = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
X_smote_all, y_smote_all = smote_wrong.fit_resample(X_all_scaled, y_all)

# THEN split — but the damage is done
X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(
    X_smote_all, y_smote_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_smote_all
)
model_wrong = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_wrong.fit(X_tr_w, y_tr_w)
y_pred_wrong = model_wrong.predict(X_te_w)
y_prob_wrong = model_wrong.predict_proba(X_te_w)[:, 1]

# Note: test set now contains synthetic samples — NOT real transactions
# Recall and AUC will look artificially inflated
recall_wrong = recall_score(y_te_w, y_pred_wrong)
auc_wrong = roc_auc_score(y_te_w, y_prob_wrong)
print(f'Recall:  {recall_wrong:.4f}  <-- Suspiciously high?')
print(f'AUC-ROC: {auc_wrong:.4f}')
print(f'Note: test set contains {y_te_w.sum()} samples labeled fraud, many are SYNTHETIC.')

print()
print('=' * 60)
print('RIGHT WAY: SMOTE applied only to training data after split')
print('=' * 60)

# RIGHT: Split first on raw data
X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE, stratify=y_all
)

# Fit scaler ONLY on training data
scaler_right = StandardScaler()
X_tr_r_scaled = scaler_right.fit_transform(X_tr_r)  # Fit on train only
X_te_r_scaled = scaler_right.transform(X_te_r)      # Apply same transform to test

# SMOTE ONLY on training data
smote_right = SMOTE(k_neighbors=5, random_state=RANDOM_STATE)
X_tr_r_smote, y_tr_r_smote = smote_right.fit_resample(X_tr_r_scaled, y_tr_r)

model_right = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
model_right.fit(X_tr_r_smote, y_tr_r_smote)

# Evaluate on REAL test set (no synthetic samples)
y_pred_right = model_right.predict(X_te_r_scaled)
y_prob_right = model_right.predict_proba(X_te_r_scaled)[:, 1]

recall_right = recall_score(y_te_r, y_pred_right)
auc_right = roc_auc_score(y_te_r, y_prob_right)
print(f'Recall:  {recall_right:.4f}  <-- Honest evaluation on real transactions')
print(f'AUC-ROC: {auc_right:.4f}')
print(f'Test set contains ONLY real transactions ({y_te_r.sum()} actual fraud cases).')
print()
print('LESSON: The "wrong way" may show better or different metrics because')
print('the model is evaluated on synthetic data it helped create — not on real unseen data.')

## Step 10: Comparing All Strategies — Side-by-Side

Now let's visualize all strategies together to understand the trade-offs.

In [ ]:
# Compile all results into a comparison DataFrame
all_metrics = {
    'Baseline': baseline_metrics,
    'Undersampling': metrics_under,
    'RandomOver': metrics_over,
    'SMOTE': metrics_smote,
    'ClassWeights': metrics_weighted,
}

# Build a clean comparison DataFrame
comparison_df = pd.DataFrame(all_metrics).T
comparison_df = comparison_df[['accuracy', 'precision', 'recall', 'f1', 'auc_roc']]
comparison_df.columns = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']

print('STRATEGY COMPARISON (all metrics for the FRAUD class):')
print(comparison_df.round(4).to_string())
print()
print('Key insight: Accuracy drops as recall improves — this is expected!')
print('The model trades false security (high accuracy) for real detection (high recall).')

In [ ]:
# ============================================================
# VISUALIZATION: Side-by-side metric comparison
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# --- LEFT: Grouped bar chart of all metrics ---
strategies = list(all_metrics.keys())
metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
metric_keys  = ['accuracy', 'precision', 'recall', 'f1', 'auc_roc']

x = np.arange(len(strategies))  # Bar group positions
width = 0.15                      # Width of each individual bar
bar_colors = ['#3498db', '#e67e22', '#e74c3c', '#2ecc71', '#9b59b6']

for i, (metric_name, metric_key, color) in enumerate(zip(metrics_names, metric_keys, bar_colors)):
    values = [all_metrics[s][metric_key] for s in strategies]
    offset = (i - 2) * width  # Center the group around x tick
    bars = axes[0].bar(x + offset, values, width, label=metric_name,
                       color=color, alpha=0.85, edgecolor='black', linewidth=0.5)

axes[0].set_title('All Metrics by Strategy (Fraud Class)', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(strategies, rotation=15, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.15)
axes[0].legend(loc='upper right', fontsize=9)
axes[0].axhline(y=1.0, color='gray', linestyle='--', alpha=0.4, linewidth=0.8)

# Add a horizontal line showing baseline accuracy
axes[0].axhline(y=baseline_metrics['accuracy'], color='#3498db',
                linestyle=':', alpha=0.6, linewidth=1.5, label='Baseline accuracy')

# --- RIGHT: Focus on Recall and F1 (the metrics that matter for fraud) ---
key_metrics = ['recall', 'f1', 'auc_roc']
key_names   = ['Recall\n(catch fraud)', 'F1-Score\n(balance)', 'AUC-ROC\n(overall rank)']
key_colors  = ['#e74c3c', '#2ecc71', '#9b59b6']

x2 = np.arange(len(strategies))
width2 = 0.25

for i, (mname, mkey, col) in enumerate(zip(key_names, key_metrics, key_colors)):
    vals = [all_metrics[s][mkey] for s in strategies]
    offset = (i - 1) * width2
    axes[1].bar(x2 + offset, vals, width2, label=mname, color=col, alpha=0.85,
               edgecolor='black', linewidth=0.5)
    # Add value labels on bars
    for j, v in enumerate(vals):
        axes[1].text(x2[j] + offset, v + 0.01, f'{v:.2f}',
                    ha='center', va='bottom', fontsize=7.5)

axes[1].set_title('Key Metrics for Fraud Detection\n(Higher = Better)',
                  fontsize=13, fontweight='bold')
axes[1].set_xticks(x2)
axes[1].set_xticklabels(strategies, rotation=15, ha='right')
axes[1].set_ylabel('Score')
axes[1].set_ylim(0, 1.15)
axes[1].legend(loc='upper left', fontsize=10)

plt.suptitle('Imbalance Handling Strategy Comparison', fontsize=15,
             fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Step 11: Confusion Matrices — Baseline vs Best Strategy

In [ ]:
# Find the strategy with best recall (most important for fraud detection)
best_strategy = max(all_metrics, key=lambda k: all_metrics[k]['recall'])
print(f'Best strategy by recall: {best_strategy}')

# Predictions for each strategy to show confusion matrix
strategy_preds = {
    'Baseline':     y_pred_baseline,
    'Undersampling': y_pred_under,
    'RandomOver':   y_pred_over,
    'SMOTE':        y_pred_smote,
    'ClassWeights': y_pred_weighted,
}

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()  # Flatten 2D array to 1D for easier indexing

for idx, (name, preds) in enumerate(strategy_preds.items()):
    cm = confusion_matrix(y_test, preds)

    # Compute recall for title annotation
    rec = recall_score(y_test, preds, zero_division=0)
    prec = precision_score(y_test, preds, zero_division=0)

    # Plot confusion matrix as heatmap
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
               xticklabels=['Predicted Legit', 'Predicted Fraud'],
               yticklabels=['Actual Legit', 'Actual Fraud'],
               ax=axes[idx], linewidths=0.5)

    axes[idx].set_title(
        f'{name}\nRecall={rec:.2f}  Precision={prec:.2f}',
        fontsize=10, fontweight='bold'
    )

    # Color-code the title background for easy comparison
    if name == best_strategy:
        axes[idx].set_title(
            f'{name} ← BEST RECALL\nRecall={rec:.2f}  Precision={prec:.2f}',
            fontsize=10, fontweight='bold', color='darkred'
        )

# Hide the 6th subplot (we only have 5 strategies)
axes[5].set_visible(False)

plt.suptitle('Confusion Matrices: All Strategies vs Test Set\n'
             '(Bottom-left = missed frauds [False Negatives] = most costly!)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

# Summary of confusion matrix for baseline
cm_base = confusion_matrix(y_test, y_pred_baseline)
print(f'\nBaseline: {cm_base[1,0]} frauds MISSED (false negatives), '
      f'{cm_base[0,1]} legit flagged (false positives)')

cm_best_key = best_strategy
cm_best = confusion_matrix(y_test, strategy_preds[cm_best_key])
print(f'{best_strategy}: {cm_best[1,0]} frauds MISSED (false negatives), '
      f'{cm_best[0,1]} legit flagged (false positives)')

## Step 12: Practical Guidance — Choosing a Strategy

| Scenario | Recommended Strategy | Reason |
|----------|---------------------|--------|
| Huge dataset (millions of rows) | Undersampling | Reduces training cost without loss |
| Small dataset | SMOTE | Can't afford to lose data; synthetic samples help |
| Mixed dataset types (numerical + categorical) | Class weights | SMOTE struggles with categorical features |
| Model supports class_weight | Class weights | Simplest, no data manipulation needed |
| Extreme imbalance (1:1000+) | SMOTE + undersampling | Combine both for very severe cases |

### Key Takeaways

1. **Never trust accuracy alone** on imbalanced data — always check recall and F1 for the minority class
2. **SMOTE only on training data** — never before splitting
3. **Recall vs Precision trade-off** — in fraud detection, recall is usually more important
4. **No single best strategy** — experiment and compare on your specific dataset
5. **Class weights are free** — always try them first before complex resampling

In [ ]:
# ============================================================
# SUMMARY TABLE: When to use which strategy
# ============================================================

summary_data = {
    'Strategy':       ['Baseline', 'Undersampling', 'RandomOversampling', 'SMOTE', 'ClassWeights'],
    'Accuracy':       [f"{all_metrics[k]['accuracy']:.3f}" for k in all_metrics],
    'Precision':      [f"{all_metrics[k]['precision']:.3f}" for k in all_metrics],
    'Recall':         [f"{all_metrics[k]['recall']:.3f}" for k in all_metrics],
    'F1-Score':       [f"{all_metrics[k]['f1']:.3f}" for k in all_metrics],
    'AUC-ROC':        [f"{all_metrics[k]['auc_roc']:.3f}" for k in all_metrics],
    'Train Size':     [
        len(y_train),
        len(y_train_under),
        len(y_train_over),
        len(y_train_smote),
        len(y_train),
    ],
    'Data Modified?': ['No', 'Reduced', 'Duplicated', 'Synthetic', 'No']
}

summary_df = pd.DataFrame(summary_data).set_index('Strategy')
print('FINAL COMPARISON TABLE')
print('=' * 90)
print(summary_df.to_string())
print()
print('RECOMMENDATION FOR FRAUD DETECTION:')
print('  1. Start with ClassWeights — free, effective, no data manipulation')
print('  2. If recall is still insufficient → try SMOTE on training data')
print('  3. For very large datasets → undersampling + class weights combined')
print('  4. Always use stratified cross-validation for final model evaluation')

---

## Self-Check Questions

Test your understanding before moving on.

---

### Question 1
You train a model on 99% negative, 1% positive data without handling imbalance. You get 99.2% accuracy. Is this a good model? How would you evaluate it properly?

<details>
<summary>Show Answer</summary>

**No — 99.2% accuracy is meaningless here.** A model that predicts "negative" for every single sample would achieve ~99% accuracy due to class distribution alone.

**How to evaluate properly:**
- Check **recall for the positive class** (class 1): if it's near 0, the model catches no positives.
- Check the **confusion matrix**: look at false negatives (FN) — missed positive cases.
- Use **F1-score** for the minority class as the primary metric.
- Use **AUC-ROC** to measure how well the model ranks positives above negatives, regardless of threshold.
- Never report only accuracy; always report class-level precision, recall, and F1.

</details>

---

### Question 2
Why must SMOTE be applied ONLY to training data and never to test data?

<details>
<summary>Show Answer</summary>

**Because applying SMOTE before the split causes data leakage:**

1. SMOTE creates synthetic samples by interpolating between real minority samples in the *entire* dataset — including samples that will later become test data.
2. Synthetic samples that are neighbors of test-set fraud cases end up in the training set.
3. The model indirectly "sees" information about test fraud patterns during training.
4. Evaluation metrics become unrealistically optimistic — the model appears better than it will perform in production.

**Additionally:** The test set should always represent real-world data distribution. Adding synthetic samples to the test set means you're evaluating on artificial data, not the true deployment scenario.

**Correct practice:** Split first → apply SMOTE only to `X_train` → evaluate on unmodified `X_test`.

</details>

---

### Question 3
You have 1,000,000 legitimate transactions and 1,000 fraud cases. Would you prefer undersampling, SMOTE, or class weights? Justify your choice.

<details>
<summary>Show Answer</summary>

**Best choice: Class weights, possibly combined with undersampling.**

**Analysis:**

- **Undersampling** to 1000:1000 means discarding 999,000 legitimate transactions — a huge loss of information that could help the model learn normal behavior patterns. However, training on 2,000 vs. 1,001,000 samples is a massive speed improvement.

- **SMOTE** would need to generate ~999,000 synthetic fraud cases from just 1,000 real ones. With such extreme imbalance (1000:1), the synthetic samples would be highly concentrated in a small region — they'd not add much diversity, and SMOTE is computationally expensive at this scale.

- **Class weights** (`class_weight='balanced'`): gives fraud ~1000x weight with no data manipulation, minimal computation overhead, uses all 1,000,000 legitimate samples to learn normal behavior, and is trivially implemented with one parameter.

**Practical recommendation:** Use `class_weight='balanced'` first. If the model trains too slowly, apply moderate undersampling (e.g., keep 100,000 legitimate samples) + class weights.

</details>

---

### Question 4
SMOTE creates synthetic samples by interpolating between real minority samples. What assumption does this violate when applied to categorical features?

<details>
<summary>Show Answer</summary>

**SMOTE assumes features are continuous and that interpolation produces valid/meaningful values.**

For categorical features, this assumption is violated:

- Example: `merchant_category` is encoded as 0=grocery, 1=electronics, 2=gas_station.
- SMOTE might interpolate between two fraud cases: merchant_category = 0 and 2 → synthetic value = 1.
- But this synthetic "1" (electronics) isn't a genuine interpolation of two fraud patterns — it's a nonsensical average of two different category codes.
- Worse, if categories are one-hot encoded (0/1), interpolation produces fractional values like 0.3 or 0.7 which don't represent any real category.

**Solutions for datasets with categorical features:**
1. Use **SMOTE-NC** (Nominal Continuous) from imblearn — handles mixed data types
2. Use **class weights** — no interpolation needed
3. Use **SMOTENC** which samples categorical values from nearest neighbors rather than interpolating

</details>